# Сравнение двух DCGAN на Fashion MNIST

Задание на 5 баллов: сравнение классического и улучшенного генераторов DCGAN.


In [ ]:
# 1. Импорты

import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.layers import (
    BatchNormalization,
    Conv2D,
    Conv2DTranspose,
    Dense,
    Dropout,
    Flatten,
    Input,
    LeakyReLU,
    Reshape,
)
from tensorflow.keras.optimizers import Adam


In [ ]:
# 2. Настройки эксперимента

SEED = 42

TRAIN_LIMIT = 40000
BATCH_SIZE = 256
EPOCHS = 20
LATENT_DIM = 100

LEARNING_RATE = 0.0002
BETA_1 = 0.5

CHECKPOINT_EPOCHS = [1, 5, 10, 15, 20]
FIXED_COUNT = 25

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


In [ ]:
# 3. Загрузка Fashion MNIST

(
    (train_images, train_labels),
    (test_images, test_labels),
) = fashion_mnist.load_data()

print("Train:", train_images.shape)
print("Test:", test_images.shape)


In [ ]:
# 4. Выбор обучающих изображений

rng = np.random.default_rng(SEED)

indices = rng.permutation(
    len(train_images)
)[:TRAIN_LIMIT]

gan_images = train_images[indices]
gan_labels = train_labels[indices]

print(
    "Изображений для GAN:",
    gan_images.shape,
)


In [ ]:
# 5. Нормализация изображений

gan_images = (
    gan_images.astype(np.float32)
    - 127.5
) / 127.5

gan_images = np.expand_dims(
    gan_images,
    axis=-1,
)

print("Форма:", gan_images.shape)
print("Минимум:", gan_images.min())
print("Максимум:", gan_images.max())


In [ ]:
# 6. Создание датасета

def create_train_dataset():

    return (
        tf.data.Dataset
        .from_tensor_slices(
            gan_images
        )
        .shuffle(
            TRAIN_LIMIT,
            seed=SEED,
            reshuffle_each_iteration=True,
        )
        .batch(
            BATCH_SIZE,
            drop_remainder=True,
        )
        .prefetch(
            tf.data.AUTOTUNE
        )
    )


train_dataset = create_train_dataset()

print(
    "Количество батчей:",
    len(train_dataset),
)


In [ ]:
# 7. Названия классов

class_names = [
    "Футболка",
    "Брюки",
    "Свитер",
    "Платье",
    "Пальто",
    "Сандалии",
    "Рубашка",
    "Кроссовки",
    "Сумка",
    "Ботинки",
]


In [ ]:
# 8. Просмотр данных

plt.figure(
    figsize=(10, 10)
)

for index in range(25):

    image = (
        gan_images[
            index,
            :,
            :,
            0,
        ]
        + 1.0
    ) / 2.0

    plt.subplot(
        5,
        5,
        index + 1,
    )

    plt.imshow(
        image,
        cmap="gray",
    )

    plt.title(
        class_names[
            gan_labels[index]
        ],
        fontsize=9,
    )

    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# 9. Проверка батча

for batch in train_dataset.take(1):

    print(
        "Форма батча:",
        batch.shape,
    )

    print(
        "Диапазон:",
        float(tf.reduce_min(batch)),
        "-",
        float(tf.reduce_max(batch)),
    )


In [ ]:
# 10. Общий дискриминатор

def create_discriminator():

    model = Sequential(
        name="Discriminator"
    )

    model.add(
        Input(
            shape=(28, 28, 1)
        )
    )

    model.add(
        Conv2D(
            64,
            kernel_size=4,
            strides=2,
            padding="same",
        )
    )

    model.add(
        LeakyReLU(
            negative_slope=0.2
        )
    )

    model.add(
        Dropout(0.3)
    )

    model.add(
        Conv2D(
            128,
            kernel_size=4,
            strides=2,
            padding="same",
        )
    )

    model.add(
        BatchNormalization()
    )

    model.add(
        LeakyReLU(
            negative_slope=0.2
        )
    )

    model.add(
        Dropout(0.3)
    )

    model.add(
        Flatten()
    )

    model.add(
        Dense(
            1,
            activation="sigmoid",
        )
    )

    return model


In [ ]:
# 11. Классический генератор

def create_classic_generator():

    model = Sequential(
        name="Classic_Generator"
    )

    model.add(
        Input(
            shape=(LATENT_DIM,)
        )
    )

    model.add(
        Dense(
            7 * 7 * 128,
            use_bias=False,
        )
    )

    model.add(
        BatchNormalization()
    )

    model.add(
        LeakyReLU(
            negative_slope=0.2
        )
    )

    model.add(
        Reshape(
            (7, 7, 128)
        )
    )

    model.add(
        Conv2DTranspose(
            128,
            kernel_size=4,
            strides=1,
            padding="same",
            use_bias=False,
        )
    )

    model.add(
        BatchNormalization()
    )

    model.add(
        LeakyReLU(
            negative_slope=0.2
        )
    )

    model.add(
        Conv2DTranspose(
            64,
            kernel_size=4,
            strides=2,
            padding="same",
            use_bias=False,
        )
    )

    model.add(
        BatchNormalization()
    )

    model.add(
        LeakyReLU(
            negative_slope=0.2
        )
    )

    model.add(
        Conv2DTranspose(
            1,
            kernel_size=4,
            strides=2,
            padding="same",
            activation="tanh",
        )
    )

    return model


In [ ]:
# 12. Улучшенный генератор

def create_improved_generator():

    model = Sequential(
        name="Improved_Generator"
    )

    model.add(
        Input(
            shape=(LATENT_DIM,)
        )
    )

    model.add(
        Dense(
            14 * 14 * 64,
            use_bias=False,
        )
    )

    model.add(
        BatchNormalization()
    )

    model.add(
        LeakyReLU(
            negative_slope=0.2
        )
    )

    model.add(
        Reshape(
            (14, 14, 64)
        )
    )

    model.add(
        Conv2D(
            128,
            kernel_size=4,
            strides=2,
            padding="same",
            use_bias=False,
        )
    )

    model.add(
        BatchNormalization()
    )

    model.add(
        LeakyReLU(
            negative_slope=0.2
        )
    )

    model.add(
        Conv2DTranspose(
            128,
            kernel_size=4,
            strides=2,
            padding="same",
            use_bias=False,
        )
    )

    model.add(
        BatchNormalization()
    )

    model.add(
        LeakyReLU(
            negative_slope=0.2
        )
    )

    model.add(
        Conv2DTranspose(
            64,
            kernel_size=4,
            strides=2,
            padding="same",
            use_bias=False,
        )
    )

    model.add(
        BatchNormalization()
    )

    model.add(
        LeakyReLU(
            negative_slope=0.2
        )
    )

    model.add(
        Conv2D(
            1,
            kernel_size=3,
            padding="same",
            activation="tanh",
        )
    )

    return model


In [ ]:
# 13. Создание моделей

tf.random.set_seed(SEED)
classic_generator = create_classic_generator()

tf.random.set_seed(SEED)
improved_generator = create_improved_generator()

classic_generator.summary()
improved_generator.summary()

classic_params = classic_generator.count_params()
improved_params = improved_generator.count_params()

print(
    "Classic Generator:",
    f"{classic_params:,}",
    "параметров",
)

print(
    "Improved Generator:",
    f"{improved_params:,}",
    "параметров",
)


In [ ]:
# 14. Фиксированный шум

fixed_noise = tf.random.normal(
    (
        FIXED_COUNT,
        LATENT_DIM,
    ),
    seed=SEED,
)


In [ ]:
# 15. Функции потерь

cross_entropy = (
    tf.keras.losses.BinaryCrossentropy()
)


def discriminator_loss(
    real_output,
    fake_output,
):

    real_loss = cross_entropy(
        tf.ones_like(
            real_output
        ),
        real_output,
    )

    fake_loss = cross_entropy(
        tf.zeros_like(
            fake_output
        ),
        fake_output,
    )

    return (
        real_loss
        + fake_loss
    )


def generator_loss(
    fake_output,
):

    return cross_entropy(
        tf.ones_like(
            fake_output
        ),
        fake_output,
    )


In [ ]:
# 16. Начальные веса дискриминатора

tf.random.set_seed(SEED)
base_discriminator = create_discriminator()

initial_discriminator_weights = (
    base_discriminator.get_weights()
)


In [ ]:
# 17. Обучение одной GAN

def train_gan(
    generator,
    model_name,
):

    dataset = create_train_dataset()

    discriminator = create_discriminator()

    discriminator.set_weights(
        initial_discriminator_weights
    )

    generator_optimizer = Adam(
        learning_rate=LEARNING_RATE,
        beta_1=BETA_1,
    )

    discriminator_optimizer = Adam(
        learning_rate=LEARNING_RATE,
        beta_1=BETA_1,
    )

    generator_history = []
    discriminator_history = []
    snapshots = {}

    @tf.function
    def train_step(real_images):

        noise = tf.random.normal(
            (
                BATCH_SIZE,
                LATENT_DIM,
            )
        )

        with tf.GradientTape() as gen_tape, \
             tf.GradientTape() as disc_tape:

            generated_images = generator(
                noise,
                training=True,
            )

            real_output = discriminator(
                real_images,
                training=True,
            )

            fake_output = discriminator(
                generated_images,
                training=True,
            )

            gen_loss = generator_loss(
                fake_output
            )

            disc_loss = discriminator_loss(
                real_output,
                fake_output,
            )

        generator_gradients = (
            gen_tape.gradient(
                gen_loss,
                generator.trainable_variables,
            )
        )

        discriminator_gradients = (
            disc_tape.gradient(
                disc_loss,
                discriminator.trainable_variables,
            )
        )

        generator_optimizer.apply_gradients(
            zip(
                generator_gradients,
                generator.trainable_variables,
            )
        )

        discriminator_optimizer.apply_gradients(
            zip(
                discriminator_gradients,
                discriminator.trainable_variables,
            )
        )

        return gen_loss, disc_loss

    print()
    print(f"Обучение: {model_name}")
    print()

    for epoch in range(
        1,
        EPOCHS + 1,
    ):

        epoch_gen_losses = []
        epoch_disc_losses = []

        for real_batch in dataset:

            gen_loss, disc_loss = train_step(
                real_batch
            )

            epoch_gen_losses.append(
                float(gen_loss)
            )

            epoch_disc_losses.append(
                float(disc_loss)
            )

        mean_gen_loss = np.mean(
            epoch_gen_losses
        )

        mean_disc_loss = np.mean(
            epoch_disc_losses
        )

        generator_history.append(
            mean_gen_loss
        )

        discriminator_history.append(
            mean_disc_loss
        )

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"G loss: {mean_gen_loss:.4f} | "
            f"D loss: {mean_disc_loss:.4f}"
        )

        if epoch in CHECKPOINT_EPOCHS:

            generated = generator(
                fixed_noise,
                training=False,
            )

            snapshots[epoch] = (
                generated.numpy()
            )

    return {
        "generator": generator,
        "discriminator": discriminator,
        "generator_loss": generator_history,
        "discriminator_loss": discriminator_history,
        "snapshots": snapshots,
    }


In [ ]:
# 18. Обучение Classic DCGAN

tf.random.set_seed(SEED)

classic_result = train_gan(
    classic_generator,
    "Classic DCGAN",
)


In [ ]:
# 19. Обучение Improved DCGAN

tf.random.set_seed(SEED)

improved_result = train_gan(
    improved_generator,
    "Improved DCGAN",
)


In [ ]:
# 20. Сравнение по эпохам

fig, axes = plt.subplots(
    len(CHECKPOINT_EPOCHS),
    10,
    figsize=(15, 12),
)

for row, epoch in enumerate(
    CHECKPOINT_EPOCHS
):

    classic_images = classic_result[
        "snapshots"
    ][epoch]

    improved_images = improved_result[
        "snapshots"
    ][epoch]

    for column in range(5):

        classic_image = (
            classic_images[
                column,
                :,
                :,
                0,
            ]
            + 1.0
        ) / 2.0

        improved_image = (
            improved_images[
                column,
                :,
                :,
                0,
            ]
            + 1.0
        ) / 2.0

        axes[
            row,
            column,
        ].imshow(
            classic_image,
            cmap="gray",
            vmin=0,
            vmax=1,
        )

        axes[
            row,
            column + 5,
        ].imshow(
            improved_image,
            cmap="gray",
            vmin=0,
            vmax=1,
        )

        axes[
            row,
            column,
        ].axis("off")

        axes[
            row,
            column + 5,
        ].axis("off")

    axes[
        row,
        0,
    ].set_ylabel(
        f"Эпоха {epoch}",
        fontsize=10,
    )

axes[0, 2].set_title(
    "Classic DCGAN",
    fontsize=14,
)

axes[0, 7].set_title(
    "Improved DCGAN",
    fontsize=14,
)

plt.suptitle(
    "Динамика генерации Fashion MNIST",
    fontsize=16,
)

plt.tight_layout()
plt.show()


In [ ]:
# 21. Финальное сравнение

classic_final = classic_result[
    "snapshots"
][EPOCHS]

improved_final = improved_result[
    "snapshots"
][EPOCHS]

fig, axes = plt.subplots(
    5,
    10,
    figsize=(15, 8),
)

for index in range(FIXED_COUNT):

    row = index // 5
    column = index % 5

    classic_image = (
        classic_final[
            index,
            :,
            :,
            0,
        ]
        + 1.0
    ) / 2.0

    improved_image = (
        improved_final[
            index,
            :,
            :,
            0,
        ]
        + 1.0
    ) / 2.0

    axes[
        row,
        column,
    ].imshow(
        classic_image,
        cmap="gray",
        vmin=0,
        vmax=1,
    )

    axes[
        row,
        column + 5,
    ].imshow(
        improved_image,
        cmap="gray",
        vmin=0,
        vmax=1,
    )

    axes[
        row,
        column,
    ].axis("off")

    axes[
        row,
        column + 5,
    ].axis("off")

axes[0, 2].set_title(
    "Classic DCGAN",
    fontsize=14,
)

axes[0, 7].set_title(
    "Improved DCGAN",
    fontsize=14,
)

plt.suptitle(
    "Сравнение после 20 эпох",
    fontsize=16,
)

plt.tight_layout()
plt.show()


In [ ]:
# 22. Графики потерь

epochs_range = range(
    1,
    EPOCHS + 1,
)

plt.figure(
    figsize=(10, 5)
)

plt.plot(
    epochs_range,
    classic_result[
        "generator_loss"
    ],
    label="Classic Generator",
)

plt.plot(
    epochs_range,
    improved_result[
        "generator_loss"
    ],
    label="Improved Generator",
)

plt.xlabel("Эпоха")
plt.ylabel("Generator loss")
plt.title("Сравнение генераторов")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(
    figsize=(10, 5)
)

plt.plot(
    epochs_range,
    classic_result[
        "discriminator_loss"
    ],
    label="Classic DCGAN",
)

plt.plot(
    epochs_range,
    improved_result[
        "discriminator_loss"
    ],
    label="Improved DCGAN",
)

plt.xlabel("Эпоха")
plt.ylabel("Discriminator loss")
plt.title("Сравнение дискриминаторов")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# 23. Данные для классификатора

classifier_train_images = (
    gan_images + 1.0
) / 2.0

classifier_train_labels = gan_labels

classifier_test_images = (
    test_images.astype(np.float32)
    / 255.0
)

classifier_test_images = np.expand_dims(
    classifier_test_images,
    axis=-1,
)


In [ ]:
# 24. Классификатор Fashion MNIST

classifier = Sequential(
    name="Fashion_Classifier"
)

classifier.add(
    Input(
        shape=(28, 28, 1)
    )
)

classifier.add(
    Conv2D(
        32,
        kernel_size=3,
        activation="relu",
        padding="same",
    )
)

classifier.add(
    Conv2D(
        64,
        kernel_size=3,
        strides=2,
        activation="relu",
        padding="same",
    )
)

classifier.add(
    Dropout(0.25)
)

classifier.add(
    Conv2D(
        128,
        kernel_size=3,
        strides=2,
        activation="relu",
        padding="same",
    )
)

classifier.add(
    Flatten()
)

classifier.add(
    Dense(
        128,
        activation="relu",
    )
)

classifier.add(
    Dropout(0.3)
)

classifier.add(
    Dense(
        10,
        activation="softmax",
    )
)

classifier.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)


In [ ]:
# 25. Обучение классификатора

classifier.fit(
    classifier_train_images,
    classifier_train_labels,
    validation_split=0.1,
    epochs=3,
    batch_size=256,
    verbose=1,
)

classifier_loss, classifier_accuracy = (
    classifier.evaluate(
        classifier_test_images,
        test_labels,
        verbose=0,
    )
)

print(
    "Test accuracy:",
    f"{classifier_accuracy:.4f}",
)


In [ ]:
# 26. Генерация тестовой выборки

EVALUATION_COUNT = 1000

comparison_noise = tf.random.normal(
    (
        EVALUATION_COUNT,
        LATENT_DIM,
    ),
    seed=2026,
)

classic_eval = classic_result[
    "generator"
](
    comparison_noise,
    training=False,
)

improved_eval = improved_result[
    "generator"
](
    comparison_noise,
    training=False,
)

classic_eval_images = (
    classic_eval.numpy()
    + 1.0
) / 2.0

improved_eval_images = (
    improved_eval.numpy()
    + 1.0
) / 2.0

classic_eval_images = np.clip(
    classic_eval_images,
    0.0,
    1.0,
)

improved_eval_images = np.clip(
    improved_eval_images,
    0.0,
    1.0,
)


In [ ]:
# 27. Оценка генераторов

classic_predictions = classifier.predict(
    classic_eval_images,
    verbose=0,
)

improved_predictions = classifier.predict(
    improved_eval_images,
    verbose=0,
)

classic_classes = np.argmax(
    classic_predictions,
    axis=1,
)

improved_classes = np.argmax(
    improved_predictions,
    axis=1,
)

classic_confidence = np.max(
    classic_predictions,
    axis=1,
)

improved_confidence = np.max(
    improved_predictions,
    axis=1,
)

classic_mean_confidence = np.mean(
    classic_confidence
)

improved_mean_confidence = np.mean(
    improved_confidence
)

classic_distribution = np.bincount(
    classic_classes,
    minlength=10,
)

improved_distribution = np.bincount(
    improved_classes,
    minlength=10,
)

MIN_CLASS_COUNT = (
    EVALUATION_COUNT
    * 0.01
)

classic_active_classes = np.sum(
    classic_distribution
    >= MIN_CLASS_COUNT
)

improved_active_classes = np.sum(
    improved_distribution
    >= MIN_CLASS_COUNT
)


In [ ]:
# 28. Графики распределения классов

plt.figure(
    figsize=(12, 5)
)

plt.bar(
    class_names,
    classic_distribution,
)

plt.xlabel("Класс")
plt.ylabel("Количество изображений")
plt.title(
    "Распределение классов — Classic DCGAN"
)

plt.xticks(
    rotation=45,
    ha="right",
)

plt.tight_layout()
plt.show()

plt.figure(
    figsize=(12, 5)
)

plt.bar(
    class_names,
    improved_distribution,
)

plt.xlabel("Класс")
plt.ylabel("Количество изображений")
plt.title(
    "Распределение классов — Improved DCGAN"
)

plt.xticks(
    rotation=45,
    ha="right",
)

plt.tight_layout()
plt.show()


In [ ]:
# 29. Итоговое сравнение

print(
    f"{'Показатель':<32}"
    f"{'Classic':>15}"
    f"{'Improved':>15}"
)

print(
    "-" * 62
)

print(
    f"{'Параметры генератора':<32}"
    f"{classic_params:>15,}"
    f"{improved_params:>15,}"
)

print(
    f"{'Generator loss':<32}"
    f"{classic_result['generator_loss'][-1]:>15.4f}"
    f"{improved_result['generator_loss'][-1]:>15.4f}"
)

print(
    f"{'Discriminator loss':<32}"
    f"{classic_result['discriminator_loss'][-1]:>15.4f}"
    f"{improved_result['discriminator_loss'][-1]:>15.4f}"
)

print(
    f"{'Средняя уверенность':<32}"
    f"{classic_mean_confidence:>15.4f}"
    f"{improved_mean_confidence:>15.4f}"
)

print(
    f"{'Представлено классов':<32}"
    f"{classic_active_classes:>15}"
    f"{improved_active_classes:>15}"
)


In [ ]:
# 30. Автоматическое сравнение

if (
    improved_mean_confidence
    > classic_mean_confidence
):

    confidence_winner = (
        "Improved DCGAN"
    )

else:

    confidence_winner = (
        "Classic DCGAN"
    )


if (
    improved_active_classes
    > classic_active_classes
):

    diversity_winner = (
        "Improved DCGAN"
    )

elif (
    improved_active_classes
    < classic_active_classes
):

    diversity_winner = (
        "Classic DCGAN"
    )

else:

    diversity_winner = (
        "Одинаковый результат"
    )


print(
    "Лучший результат по уверенности:",
    confidence_winner,
)

print(
    "Лучший результат по разнообразию:",
    diversity_winner,
)


In [ ]:
# 31. Сохранение моделей

classic_generator.save(
    "classic_dcgan_generator.keras"
)

improved_generator.save(
    "improved_dcgan_generator.keras"
)

print(
    "Модели сохранены."
)


In [ ]:
# 32. Выводы

print()
print("Выводы")
print()
print(
    "В работе были сравнены классический и улучшенный "
    "генераторы DCGAN на Fashion MNIST."
)
print(
    "Обе модели обучались в одинаковых условиях на 40 000 "
    "изображений в течение 20 эпох."
)
print(
    "Сравнение выполнено по визуальному качеству, динамике "
    "loss, средней уверенности классификатора и разнообразию классов."
)
print(
    "Окончательное преимущество модели следует оценивать по "
    "совокупности полученных визуальных и численных результатов."
)
